[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/05-explainability/04-term_fields_explained.ipynb)

In [1]:
# !pip install mbox

# TERM Fields Explained

`IndexType.TERM` is meant for single-word values: categories, city names, statuses. On data that actually looks like that, `TERM` behaves a lot like `PHRASE`, typo-tolerant, with a slightly wider cutoff than `IDENT`. The previous notebook's whole point, though, was that `PHRASE` stops comparing character-by-character once a value has more than one word. `TERM` never stops. If you hand a `TERM` field a multi-word value, on purpose or by a schema mistake, it keeps doing a single, whole-string character comparison, and the results look nothing like what you just saw for `PHRASE`.

In this notebook you will:

1. Confirm `TERM` behaves like a typo-tolerant match on the single-word data it is meant for
2. Rerun the exact scenarios from `03-phrase_fields_explained.ipynb`, this time on a `TERM`-typed field, and see the numbers actually diverge
3. Learn that `COMPLETE` on `TERM` is a fuzzy *prefix* match, not a substring search
4. See that `DETECT` has no such anchoring restriction
5. Confirm `EXACT` normalizes case but not whitespace, one degree less forgiving than `PHRASE`'s `EXACT`
6. Walk away with a clear rule for when `TERM` is the right choice, and when it silently isn't

In [2]:
import pandas as pd
from mbox.indexing import TableIndexer
from mbox.recall import TableRecallMode

catalog = pd.read_csv("datasets/product_catalog.csv")
index = TableIndexer.create_index(
    catalog,
    index_columns=["product_id", "product_name", "category", "year_released", "price"],
    tmp_dir="tmp_index"
)

catalog[["product_id", "category"]]

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


,product_id,category
0,B88-EXT-24,Batteries
1,A12-PWR-22,Batteries
2,C99-SNS-23,Cameras
3,D45-REL-21,Automation
4,E67-LEN-24,Cameras
5,F31-TRK-20,Automation
6,G14-CAB-19,Accessories
7,H82-DOC-23,Accessories
8,I29-BAT-22,Batteries
9,J56-SEN-24,Automation


`category` is a `TERM` field, inferred automatically, genuinely single-word values: `"Batteries"`, `"Cameras"`, `"Automation"`, `"Accessories"`. This is the data `TERM` is designed for.

## 1. On real single-word data, `TERM` is a typo-tolerant match

Nothing surprising here, a category name with a couple of characters off still finds its match, with a graded score.

In [3]:
category_queries = ["Batteries", "Baterries", "Automaton", "Acessories", "Camras"]

rows = []
for query in category_queries:
    result = index.match(category=query, modes={"category": TableRecallMode.APPROX}, include_field_scores=True, min_total_match_value=0, max_results=1)
    found = len(result) > 0 and result["index_row"].iloc[0] != -1
    rows.append({"query": query, "found": found, "category_score": result["category_score"].iloc[0] if found else None})

pd.DataFrame(rows)

,query,found,category_score
0,Batteries,True,100
1,Baterries,True,55
2,Automaton,True,81
3,Acessories,True,84
4,Camras,True,69


## 2. The same scenarios as `03-phrase_fields_explained.ipynb`, on a `TERM` field instead

`product_name` was inferred as `PHRASE`. To isolate what changes when the *only* thing that's different is the `IndexType`, here it is indexed a second time, forced to `TERM`, same values, same queries used in the previous notebook.

In [4]:
from mbox.config import TableConfig, TableFieldConfig, IndexType

term_name_df = pd.DataFrame({"product_name": catalog["product_name"]})
term_name_config = TableConfig(fields=[TableFieldConfig(column="product_name", index_type=IndexType.TERM)])
term_name_index = TableIndexer.create_index(df=term_name_df, config_overrides=term_name_config, tmp_dir="tmp_index")

scenarios = [
    ("full match", "Extended Battery Pack Pro"),
    ("reordered", "Battery Extended Pack Pro"),
    ("one word missing", "Extended Battery Pro"),
    ("one word added", "Extended Battery Pack Pro Max"),
    ("only one word", "Battery"),
]

rows = []
for label, query in scenarios:
    result = term_name_index.match(product_name=query, modes={"product_name": TableRecallMode.APPROX}, include_field_scores=True, min_total_match_value=0, max_results=1)
    found = len(result) > 0 and result["index_row"].iloc[0] != -1
    rows.append({"scenario": label, "query": query, "found": found, "TERM_score": result["product_name_score"].iloc[0] if found else None})

pd.DataFrame(rows)

config_overrides is given, therefore all information from index_columns, index_types, alias_sets and character_mappings is ignored.


,scenario,query,found,TERM_score
0,full match,Extended Battery Pack Pro,True,100.0
1,reordered,Battery Extended Pack Pro,False,NaN
2,one word missing,Extended Battery Pro,True,69.0
3,one word added,Extended Battery Pack Pro Max,True,77.0
4,only one word,Battery,False,NaN


Put this next to `03-phrase_fields_explained.ipynb`'s Sections 1 and 2 and the divergence is stark:

| Scenario | `PHRASE` score | `TERM` score |
|---|---|---|
| Reordered | `100` | not found at all |
| One word missing | `96` | `69` |
| One word added | `69` | `77` |
| Only one word | `89` | not found at all |

`TERM` is treating `"Extended Battery Pack Pro"` as one 26-character sequence, not four words. Reordering it scrambles character positions across the entire string, past the cutoff, exactly the way reordering `"APPROXIMATELY"`'s letters would. `PHRASE` and `TERM`'s edit-tolerance looked almost identical back in `01-index_types_and_recall_modes.ipynb`, because that test used a single word, where there's no structure for the two `IndexType`s to disagree about. The moment there's more than one word, the two diverge completely.

## 3. `COMPLETE` on `TERM` is a fuzzy prefix match, not a substring search

`03-phrase_fields_explained.ipynb` showed `COMPLETE` on `PHRASE` finding any word, or combination of words, anywhere in the value. On `TERM`, `COMPLETE` only finds fragments that start at, or very near, the beginning of the indexed string.

In [5]:
complete_queries = [
    ("prefix", "Extended Battery"),
    ("prefix, off by one character", "xtended Battery Pack Pr"),
    ("suffix", "Pack Pro"),
    ("middle, unanchored", "Battery Pack"),
    ("middle, unanchored, single word", "Battery"),
]

rows = []
for label, query in complete_queries:
    result = term_name_index.match(product_name=query, modes={"product_name": TableRecallMode.COMPLETE}, include_field_scores=True, min_total_match_value=0, max_results=1)
    found = len(result) > 0 and result["index_row"].iloc[0] != -1
    rows.append({"scenario": label, "query": query, "found": found, "TERM_score": result["product_name_score"].iloc[0] if found else None})

pd.DataFrame(rows)

,scenario,query,found,TERM_score
0,prefix,Extended Battery,True,92.0
1,"prefix, off by one character",xtended Battery Pack Pr,True,85.0
2,suffix,Pack Pro,False,NaN
3,"middle, unanchored",Battery Pack,False,NaN
4,"middle, unanchored, single word",Battery,False,NaN


The prefix and the near-prefix both find a match. The suffix and the two unanchored middle fragments do not, even though `"Battery Pack"` is a longer, more specific fragment than the prefix `"Extended Battery"` is. Position, not length or specificity, is what `COMPLETE` on `TERM` actually checks. If you need genuine "does this substring appear anywhere" behavior on a multi-word value, this is not it, `PHRASE`'s `COMPLETE` is.

## 4. `DETECT` has no such restriction

`DETECT` checks the reverse direction, whether the indexed value appears inside the query, and it is not anchored the way `COMPLETE` is. The indexed value can appear at the start, the end, or the middle of the query and still be found.

In [6]:
detect_queries = [
    "Extended Battery Pack Pro plus extra stuff at the end",
    "prefix stuff then Extended Battery Pack Pro",
    "Extended Battery Pack Pro",
]

rows = []
for query in detect_queries:
    result = term_name_index.match(product_name=query, modes={"product_name": TableRecallMode.DETECT}, include_field_scores=True, min_total_match_value=0, max_results=1)
    found = len(result) > 0 and result["index_row"].iloc[0] != -1
    rows.append({"query": query, "found": found, "TERM_score": result["product_name_score"].iloc[0] if found else None})

pd.DataFrame(rows)

,query,found,TERM_score
0,Extended Battery Pack Pro plus extra stuff at ...,True,85
1,prefix stuff then Extended Battery Pack Pro,True,88
2,Extended Battery Pack Pro,True,100


All three are found, regardless of where the indexed value sits inside the query. The anchoring behavior in Section 3 is specific to `COMPLETE`, not a general property of `TERM`.

## 5. `EXACT`: case-insensitive, but not whitespace-forgiving

`03-phrase_fields_explained.ipynb` showed `PHRASE`'s `EXACT` normalizing away case *and* doubled or surrounding whitespace before checking equality. `TERM`'s `EXACT` only takes care of case.

In [7]:
exact_queries = [
    "Extended Battery Pack Pro",
    "extended battery pack pro",
    "  Extended Battery Pack Pro ",
    "Extended  Battery Pack Pro",
]

rows = []
for query in exact_queries:
    result = term_name_index.match(product_name=query, modes={"product_name": TableRecallMode.EXACT}, include_field_scores=True, min_total_match_value=0, max_results=1)
    found = len(result) > 0 and result["index_row"].iloc[0] != -1
    rows.append({"query": repr(query), "found": found, "TERM_score": result["product_name_score"].iloc[0] if found else None})

pd.DataFrame(rows)

,query,found,TERM_score
0,'Extended Battery Pack Pro',True,100.0
1,'extended battery pack pro',True,100.0
2,' Extended Battery Pack Pro ',False,NaN
3,'Extended Battery Pack Pro',False,NaN


Case is still normalized, but leading, trailing, or doubled whitespace is not, both fail under `EXACT` here, where they would have passed on a `PHRASE` field. One more small way the two `IndexType`s are not interchangeable, even on modes that sound identical.

## 6. Practical guidance for `TERM` fields

**Use `TERM` when a value is genuinely one word or token.** Categories, city names, statuses, single-term codes. On data like that, it behaves almost identically to `PHRASE`, and slightly more tolerantly than `IDENT`.

**Do not use `TERM` expecting `PHRASE`'s reordering tolerance.** If your data can plausibly arrive in a different word order, a first-name/last-name swap, an address written two different ways, `TERM` will not forgive that the way `PHRASE` does. Section 2 is the direct evidence: identical data, identical query, and reordering alone took the score from `100` to not found at all.

**`COMPLETE` on `TERM` is closer to "starts with" than "contains."** It's a reasonable choice for autocomplete-style prefix matching on a single-word or code-like field. It is the wrong choice if what you actually need is "this fragment appears somewhere in the value," regardless of position, that is `PHRASE`'s `COMPLETE`.

**If a column keeps getting auto-inferred as `TERM` but your data is genuinely multi-word, override it.** A free-text column with unusually few unique values can be inferred as `TERM` by mistake. `03-index-configuration/01-defining_explicit_field_configs.ipynb` shows how to force it to `PHRASE` explicitly with `TableFieldConfig`, and Section 2 above is exactly what is silently at stake if you don't.

## Next steps

- **`05-ident_fields_explained.ipynb`** - identifiers, where even `TERM`'s single-word tolerance gets deliberately tightened further
- **`06-numeric_fields_explained.ipynb`** - what happens once the field is a number instead of a string
- **`07-multi_field_weights_explained.ipynb`** - combining fields of different `IndexType`s into one `overall_score`

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*